# ARXML Parser for Enums
This notebook parses ARXML files, extracts data type mappings, and generates a DataFrame with enumerated states.

In [1]:
# Import required libraries
import pandas as pd
from lxml import etree
import re
import os
import math
from typing import Any, Dict

In [ ]:
# Define the parser function
def parse_arxml_with_enums(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (This might take a moment to build lookup tables)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    # 1. Extract all CompuMethods (The actual Enum dictionaries)
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        enums = {}
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            lower_limit = scale.xpath("*[local-name()='LOWER-LIMIT']")
            vt = scale.xpath(".//*[local-name()='VT']")
            if lower_limit and vt and lower_limit[0].text and vt[0].text:
                enums[lower_limit[0].text.strip()] = vt[0].text.strip()
        
        if enums:
            compu_methods[cm_name] = enums

    # 2. Extract Application Data Types to map them to CompuMethods
    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            ref_name = compu_ref[0].text.split("/")[-1].strip()
            app_to_compu[app_name] = ref_name

    # Helper function to format the dictionary into a readable string
    def get_enum_string(app_type_name):
        compu_name = app_to_compu.get(app_type_name)
        enum_dict = compu_methods.get(compu_name, {})
        if not enum_dict:
            return "No Enums"
        return " | ".join([f"{k}: {v}" for k, v in enum_dict.items()])

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem:
            continue
        
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match:
            continue
        
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        valid_methods = []
        valuestate_app_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    continue 
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw))
        
        for clean_method, raw_app_name in valid_methods:
            base_enums = get_enum_string(raw_app_name)
            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "Enums": base_enums
            })
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                vs_enums = get_enum_string(valuestate_app_name)
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "Enums": vs_enums
                })

    df = pd.DataFrame(parsed_data)
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
    
    return df

In [ ]:
def parse_arxml_with_boundaries(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Building Enum and Physical Range tables)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        
        # Calculate Mid and round down to the lower base using math.floor
        mid_val = None
        if has_limits:
            mid_val = math.floor((min_val + max_val) / 2)
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val
        }

    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            ref_name = compu_ref[0].text.split("/")[-1].strip()
            app_to_compu[app_name] = ref_name

    def get_compu_data(app_type_name):
        compu_name = app_to_compu.get(app_type_name)
        return compu_methods.get(compu_name, {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None
        })

    def format_val(v):
        if v is None: return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem:
            continue
            
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match:
            continue
            
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        valid_methods = []
        valuestate_app_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    continue 
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw))
        
        for clean_method, raw_app_name in valid_methods:
            c_data = get_compu_data(raw_app_name)
            
            if c_data["has_enums"]:
                states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()])
            elif c_data["min"] is not None:
                states_str = "Physical Value"
            else:
                states_str = "No Data"

            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "Enums": states_str,
                "Min": format_val(c_data["min"]),
                "Mid": format_val(c_data["mid"]),
                "Max": format_val(c_data["max"])
            })
            
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                
                vs_c_data = get_compu_data(valuestate_app_name)
                vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "Enums": vs_states,
                    "Min": format_val(vs_c_data["min"]),
                    "Mid": format_val(vs_c_data["mid"]),
                    "Max": format_val(vs_c_data["max"])
                })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
        
        # --- Analytics Summary ---
        physical_count = (df['Enums'] == "Physical Value").sum()
        no_data_count = (df['Enums'] == "No Data").sum()
        enum_count = len(df) - physical_count - no_data_count
        
        print("\n--- Parsing Summary ---")
        print(f"Total Signals Extracted : {len(df)}")
        print(f"Signals with Enums      : {enum_count}")
        print(f"Physical Value Signals  : {physical_count}")
        print(f"Signals with No Data    : {no_data_count}\n")
    
    return df

In [ ]:
from IPython.display import display

# Example usage in notebook
file_name = "ETH_CAN.arxml"  # Replace with your ARXML file path
df_signals = parse_arxml_with_boundaries(file_name)

if not df_signals.empty:
    print(f"Success! Extracted {len(df_signals)} signals with Enums and Physical Boundaries.")
    
    # Configure pandas to not truncate our dataframe view
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 60)
    
    # Define the columns we want to inspect
    cols_to_view = ['Event', 'Method', 'Enums', 'Min', 'Mid', 'Max']
    
    display(df_signals[cols_to_view].head(25))
    
    # Optionally export to CSV
    # df_signals.to_csv("signals_with_boundaries.csv", index=False)
    # print("\nData exported to signals_with_boundaries.csv")
else:
    print("No signals found or failed to parse the file.")

Parsing ETH_CAN.arxml (Building Enum and Physical Range tables)...

--- Parsing Summary ---
Total Signals Extracted : 7865
Signals with Enums      : 4620
Physical Value Signals  : 1371
Signals with No Data    : 1874

Success! Extracted 7865 signals with Enums and Physical Boundaries.


,Event,Method,Available_States,Min,Mid,Max
0,SomeIpGadeSignal,GadeStatus,0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_ON_BOARD | ...,0,3,7
1,SomeIpGadeSignal,GadeStatusValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
2,SomeIpGadeSignal,gadeEvent,No Data,N/A,N/A,N/A
3,SomeIpGadeSignal,gadeEventValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
4,SomeIpMinVoltageReq,VoltageValueType,Physical Value,10,13,16
5,SomeIpMinVoltageReq,VoltageValueTypeValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
6,SomeIpMinVoltageReq,minVoltageValue,Physical Value,10,13,16
7,SomeIpMinVoltageReq,minVoltageValue1,Physical Value,10,13,16
8,SomeIpMinVoltageReq,minVoltageValue1ValueState,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2
9,SomeIpMinVoltageReq,minVoltageValue2,Physical Value,10,13,16


In [ ]:
import os
import re
import math
import pandas as pd
from lxml import etree
from IPython.display import display

def parse_arxml_with_scaling(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Extracting Enums, Ranges, Scaling, and Units)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        # 1. Extract Unit
        unit_ref = cm.xpath("*[local-name()='UNIT-REF']")
        unit = unit_ref[0].text.split("/")[-1].strip() if unit_ref and unit_ref[0].text else "N/A"
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        factor = "N/A"
        offset = "N/A"
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            coeffs_node = scale.xpath(".//*[local-name()='COMPU-RATIONAL-COEFFS']")
            
            # Min / Max Boundaries
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            # Enums
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
                
            # 2. Extract Scaling Coefficients (Factor & Offset)
            if coeffs_node:
                try:
                    num_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-NUMERATOR']/*[local-name()='V']")
                    den_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-DENOMINATOR']/*[local-name()='V']")
                    
                    # AUTOSAR specifies Numerator[0] is offset, Numerator[1] is factor
                    n0 = float(num_v[0].text) if len(num_v) > 0 else 0.0
                    n1 = float(num_v[1].text) if len(num_v) > 1 else 1.0
                    d = float(den_v[0].text) if len(den_v) > 0 else 1.0
                    
                    if d != 0:
                        offset = n0 / d
                        factor = n1 / d
                except Exception:
                    pass
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        
        mid_val = None
        if has_limits:
            mid_val = math.floor((min_val + max_val) / 2)
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val,
            "unit": unit,
            "factor": factor,
            "offset": offset
        }

    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            ref_name = compu_ref[0].text.split("/")[-1].strip()
            app_to_compu[app_name] = ref_name

    def get_compu_data(app_type_name):
        compu_name = app_to_compu.get(app_type_name)
        return compu_methods.get(compu_name, {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None,
            "unit": "N/A", "factor": "N/A", "offset": "N/A"
        })

    def format_val(v):
        if v is None or v == "N/A": return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem:
            continue
            
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match:
            continue
            
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        valid_methods = []
        valuestate_app_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    continue 
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw))
        
        for clean_method, raw_app_name in valid_methods:
            c_data = get_compu_data(raw_app_name)
            
            if c_data["has_enums"]:
                states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()])
            elif c_data["min"] is not None:
                states_str = "Physical Value"
            else:
                states_str = "No Data"

            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "Enums": states_str,
                "Min": format_val(c_data["min"]),
                "Mid": format_val(c_data["mid"]),
                "Max": format_val(c_data["max"]),
                "Factor": format_val(c_data["factor"]),
                "Offset": format_val(c_data["offset"]),
                "Unit": c_data["unit"]
            })
            
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                
                vs_c_data = get_compu_data(valuestate_app_name)
                vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "Enums": vs_states,
                    "Min": format_val(vs_c_data["min"]),
                    "Mid": format_val(vs_c_data["mid"]),
                    "Max": format_val(vs_c_data["max"]),
                    "Factor": format_val(vs_c_data["factor"]),
                    "Offset": format_val(vs_c_data["offset"]),
                    "Unit": vs_c_data["unit"]
                })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
        
        physical_count = (df['Enums'] == "Physical Value").sum()
        no_data_count = (df['Enums'] == "No Data").sum()
        enum_count = len(df) - physical_count - no_data_count
        
        print("\n--- Parsing Summary ---")
        print(f"Total Signals Extracted : {len(df)}")
        print(f"Signals with Enums      : {enum_count}")
        print(f"Physical Value Signals  : {physical_count}")
        print(f"Signals with No Data    : {no_data_count}\n")
    
    return df

# ==========================================
# EXECUTION BLOCK
# ==========================================
file_name = "ETH_CAN.arxml" 
df_signals = parse_arxml_with_scaling(file_name)

if not df_signals.empty:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 50)
    
    # Adding Factor, Offset, and Unit to our view
    cols_to_view = ['Method', 'Enums', 'Min', 'Mid', 'Max', 'Factor', 'Offset', 'Unit']
    display(df_signals[cols_to_view].head(25))

Parsing ETH_CAN.arxml (Extracting Enums, Ranges, Scaling, and Units)...


KeyboardInterrupt: 

In [ ]:
import os
import re
import math
import pandas as pd
from lxml import etree
from IPython.display import display

def parse_arxml_complete(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Cleaning up Data Types)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    # 1. CompuMethods (Enums, Min/Max/Mid, Units, Scaling)
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        unit_ref = cm.xpath("*[local-name()='UNIT-REF']")
        unit = unit_ref[0].text.split("/")[-1].strip() if unit_ref and unit_ref[0].text else "N/A"
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        factor = "N/A"
        offset = "N/A"
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            coeffs_node = scale.xpath(".//*[local-name()='COMPU-RATIONAL-COEFFS']")
            
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
                
            if coeffs_node:
                try:
                    num_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-NUMERATOR']/*[local-name()='V']")
                    den_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-DENOMINATOR']/*[local-name()='V']")
                    
                    n0 = float(num_v[0].text) if len(num_v) > 0 else 0.0
                    n1 = float(num_v[1].text) if len(num_v) > 1 else 1.0
                    d = float(den_v[0].text) if len(den_v) > 0 else 1.0
                    
                    if d != 0:
                        offset = n0 / d
                        factor = n1 / d
                except Exception: pass
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        mid_val = math.floor((min_val + max_val) / 2) if has_limits else None
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val,
            "unit": unit,
            "factor": factor,
            "offset": offset
        }

    # 2. Application -> CompuMethod Mapping
    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            app_to_compu[app_name] = compu_ref[0].text.split("/")[-1].strip()

    # 3. Implementation -> BaseType Mapping (WITH FIX)
    impl_to_basetype = {}
    for impl_dt in root.xpath("//*[local-name()='IMPLEMENTATION-DATA-TYPE']"):
        impl_name_elem = impl_dt.xpath("*[local-name()='SHORT-NAME']")
        base_ref = impl_dt.xpath(".//*[local-name()='BASE-TYPE-REF']")
        if impl_name_elem and base_ref and base_ref[0].text:
            impl_name = impl_name_elem[0].text.strip()
            basetype_raw = base_ref[0].text.split("/")[-1].strip()
            
            # --- THE FIX ---
            # Clean up AUTOSAR authoring tool quirks (e.g., 'uint81' -> 'uint8', 'float32_0' -> 'float32')
            clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', basetype_raw, re.IGNORECASE)
            basetype_name = clean_match.group(1).lower() if clean_match else basetype_raw
            
            impl_to_basetype[impl_name] = basetype_name

    def get_compu_data(app_type_name):
        return compu_methods.get(app_to_compu.get(app_type_name), {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None,
            "unit": "N/A", "factor": "N/A", "offset": "N/A"
        })

    def format_val(v):
        if v is None or v == "N/A": return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
            
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match: continue
            
        sif = match.group(1)
        someip_event = f"SomeIp{match.group(2)}"
        
        valid_methods = []
        valuestate_app_name = None
        valuestate_impl_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            impl_ref = dt_map.xpath("*[local-name()='IMPLEMENTATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                impl_path_raw = impl_ref[0].text.split("/")[-1].strip() if impl_ref and impl_ref[0].text else None
                
                # Check for ValueState placeholders
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    valuestate_impl_name = impl_path_raw # Save implementation name for data type lookup
                    continue 
                    
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw, impl_path_raw))
        
        for clean_method, raw_app_name, raw_impl_name in valid_methods:
            c_data = get_compu_data(raw_app_name)
            datatype = impl_to_basetype.get(raw_impl_name, "N/A") if raw_impl_name else "N/A"
            
            if c_data["has_enums"]:
                states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()])
            elif c_data["min"] is not None:
                states_str = "Physical Value"
            else:
                states_str = "No Data"

            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "DataType": datatype,
                "Enums": states_str,
                "Min": format_val(c_data["min"]),
                "Mid": format_val(c_data["mid"]),
                "Max": format_val(c_data["max"]),
                "Factor": format_val(c_data["factor"]),
                "Offset": format_val(c_data["offset"]),
                "Unit": c_data["unit"]
            })
            
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                
                vs_c_data = get_compu_data(valuestate_app_name)
                vs_datatype = impl_to_basetype.get(valuestate_impl_name, "N/A") if valuestate_impl_name else "N/A"
                vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "DataType": vs_datatype,
                    "Enums": vs_states,
                    "Min": format_val(vs_c_data["min"]),
                    "Mid": format_val(vs_c_data["mid"]),
                    "Max": format_val(vs_c_data["max"]),
                    "Factor": format_val(vs_c_data["factor"]),
                    "Offset": format_val(vs_c_data["offset"]),
                    "Unit": vs_c_data["unit"]
                })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
    
    return df

# ==========================================
# EXECUTION BLOCK
# ==========================================
file_name = "ETH_CAN.arxml" 
df_signals = parse_arxml_complete(file_name)

if not df_signals.empty:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 40)
    
    cols_to_view = ['Method', 'DataType', 'Enums', 'Min', 'Mid', 'Max', 'Unit']
    display(df_signals[cols_to_view].head(25))

Parsing ETH_CAN.arxml (Cleaning up Data Types)...


,Method,DataType,Available_States,Min,Mid,Max,Unit
0,GadeStatus,uint8,0: GADESTATUS_PARK_MODE | 1: GADESTA...,0,3,7,N/A
1,GadeStatusValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
2,gadeEvent,uint8,No Data,N/A,N/A,N/A,N/A
3,gadeEventValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
4,VoltageValueType,float32,Physical Value,10,13,16,Volt
5,VoltageValueTypeValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
6,minVoltageValue,float32,Physical Value,10,13,16,Volt
7,minVoltageValue1,float32,Physical Value,10,13,16,Volt
8,minVoltageValue1ValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,1,2,N/A
9,minVoltageValue2,float32,Physical Value,10,13,16,Volt


In [ ]:
# 1. Overall Signal Categorization
physical_count = (df_signals['Enums'] == "Physical Value").sum()
no_data_count = (df_signals['Enums'] == "No Data").sum()
enum_count = len(df_signals) - physical_count - no_data_count

print("=== PARSING SUMMARY ===")
print(f"Total Signals       : {len(df_signals)}")
print(f"Signals with Enums  : {enum_count}")
print(f"Physical Signals    : {physical_count}")
print(f"Missing / No Data   : {no_data_count}\n")

# 2. Data Type Distribution (This will prove if 'double' was caught!)
print("=== DATA TYPE DISTRIBUTION ===")
print(df_signals['DataType'].value_counts())
print("\n")

# 3. Quick check on how many physical signals have Units defined
print("=== UNIT DISTRIBUTION ===")
# Filter out "N/A" to see the actual units being used
print(df_signals[df_signals['Unit'] != "N/A"]['Unit'].value_counts())
print("\n")

# 4. View a sample of the "No Data" signals to investigate them
print("=== NO DATA SAMPLE ===")
df_no_data = df_signals[df_signals['Enums'] == "No Data"]
display(df_no_data[['Event', 'Method', 'DataType']].head(10))

=== PARSING SUMMARY ===
Total Signals       : 7865
Signals with Enums  : 4620
Physical Signals    : 1371
Missing / No Data   : 1874

=== DATA TYPE DISTRIBUTION ===
DataType
uint8      6291
float32     513
uint16      275
uint32      250
N/A         226
boolean     136
uint64       55
sint32       45
sint16       32
sint64       26
sint8        10
UTF_8         6
Name: count, dtype: int64


=== UNIT DISTRIBUTION ===
Unit
CELSIUS       47
X_3           44
PERCENTAGE    43
V1            30
m1            28
              ..
l___mn         1
L___min        1
km___h         1
m2___s         1
X1___m         1
Name: count, Length: 100, dtype: int64


=== NO DATA SAMPLE ===


,Event,Method,DataType
2,SomeIpGadeSignal,gadeEvent,uint8
12,SomeIpMinVoltageReq,setMinVoltageAC,uint8
14,SomeIpMinVoltageReq,setMinVoltageWashing,uint8
16,SomeIpMinVoltageReq,setMinVoltageWiping,uint8
33,SomeIpPrimBattVoltRegulStatus,dcdcFaultypeEvent,uint8
34,SomeIpPrimBattVoltRegulStatus,dcdcStateEvent,uint8
36,SomeIpPrimBattVoltRegulStatus,dcdcTemperatureInfoEvent,uint8
38,SomeIpPrimBattVoltRegulStatus,voltageRegulationInfoEvent,uint8
39,SomeIpPrimBattVoltRegulStatus,voltageRegulationStateEvent,uint8
42,SomeIpPwtProducerLoad,lifeOnBoardPowerLimitationEvent,uint8


In [ ]:
# 1. Force Pandas to show ALL columns and expand column width for readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60) 

# 2. Stitch the first 15 and last 15 rows together into a single view
display(pd.concat([df_signals.head(15), df_signals.tail(15)]))

# ---------------------------------------------------------
# ALTERNATIVE: If you prefer to view them as two completely separate tables
# ---------------------------------------------------------
# print("--- First 15 Signals ---")
# display(df_signals.head(15))

# print("--- Last 15 Signals ---")
# display(df_signals.tail(15))

,Cluster,SIF,Event,Method,Signal_String,DataType,Available_States,Min,Mid,Max,Factor,Offset,Unit
0,EthernetCluster,16414,SomeIpGadeSignal,GadeStatus,"""EthernetCluster::sif_16414::SomeIpGadeSignal::GadeStatus""",uint8,0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_ON_BOARD | ...,0,3,7,N/A,N/A,N/A
1,EthernetCluster,16414,SomeIpGadeSignal,GadeStatusValueState,"""EthernetCluster::sif_16414::SomeIpGadeSignal::GadeStatu...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
2,EthernetCluster,16414,SomeIpGadeSignal,gadeEvent,"""EthernetCluster::sif_16414::SomeIpGadeSignal::gadeEvent""",uint8,No Data,N/A,N/A,N/A,N/A,N/A,N/A
3,EthernetCluster,16414,SomeIpGadeSignal,gadeEventValueState,"""EthernetCluster::sif_16414::SomeIpGadeSignal::gadeEvent...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
4,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueType,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::Voltag...",float32,Physical Value,10,13,16,1,0,Volt
5,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueTypeValueState,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::Voltag...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
6,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",float32,Physical Value,10,13,16,1,0,Volt
7,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",float32,Physical Value,10,13,16,1,0,Volt
8,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1ValueState,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: V...,0,1,2,N/A,N/A,N/A
9,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue2,"""EthernetCluster::sif_16422::SomeIpMinVoltageReq::minVol...",float32,Physical Value,10,13,16,1,0,Volt


In [ ]:
# 1. Remove exact duplicate rows across all columns
# df = df_signals.drop_duplicates()

# (Optional: If you want to ensure uniqueness specifically based on the Signal String itself)
df = df_signals.drop_duplicates(subset=['Signal_String'])

# 2. Count how many signals have Enums vs. how many don't
# Note: Our script explicitly labeled missing enums as the string "No Enums"
missing_enums_count = (df['Enums'] == "No Enums").sum()
has_enums_count = (df['Enums'] != "No Enums").sum()

print(f"Total Unique Signals: {len(df)}")
print(f"Signals WITH Enums: {has_enums_count}")
print(f"Signals WITHOUT Enums (False Positives/Missing): {missing_enums_count}")

# 3. (Bonus) View a quick breakdown of exactly which enums are present
# This shows the top 10 most frequent enum states across your database
print("\n--- Top 10 Most Common Enum Mappings ---")
print(df['Enums'].value_counts().head(10))

Total Unique Signals: 7795
Signals WITH Enums: 4558
Signals WITHOUT Enums (False Positives/Missing): 3237

--- Top 10 Most Common Enum Mappings ---
Available_States
0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: VALUE_STATE_INVALID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  3374
No Enums                                                                                                                                          

In [ ]:
# Display a dataframe containing ONLY the signals missing their Enums
df_missing = df[df['Enums'] == "No Enums"]
df_missing

,Cluster,SIF,Event,Method,Signal_String,Available_States
2,EthernetCluster,16414,SomeIpGadeSignal,gadeEvent,"""EthernetCluster::sif_16414::SomeIpGadeSignal:...",No Enums
4,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueType,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
6,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
7,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
9,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue2,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
...,...,...,...,...,...,...
7855,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,Uuid6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7857,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,lsb6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7859,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,msb6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7861,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,setRHLBodyCmdRsp,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums


In [ ]:
import os
import re
import math
import pandas as pd
from lxml import etree
from IPython.display import display

def parse_arxml_complete(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Extracting Real Ports & Cleaning Types)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    # 1. CompuMethods (Enums, Min/Max/Mid, Units, Scaling)
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem: continue
        cm_name = cm_name_elem[0].text.strip()
        
        unit_ref = cm.xpath("*[local-name()='UNIT-REF']")
        unit = unit_ref[0].text.split("/")[-1].strip() if unit_ref and unit_ref[0].text else "N/A"
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        factor = "N/A"
        offset = "N/A"
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            coeffs_node = scale.xpath(".//*[local-name()='COMPU-RATIONAL-COEFFS']")
            
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
                
            if coeffs_node:
                try:
                    num_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-NUMERATOR']/*[local-name()='V']")
                    den_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-DENOMINATOR']/*[local-name()='V']")
                    
                    n0 = float(num_v[0].text) if len(num_v) > 0 else 0.0
                    n1 = float(num_v[1].text) if len(num_v) > 1 else 1.0
                    d = float(den_v[0].text) if len(den_v) > 0 else 1.0
                    
                    if d != 0:
                        offset = n0 / d
                        factor = n1 / d
                except Exception: pass
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        mid_val = math.floor((min_val + max_val) / 2) if has_limits else None
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val,
            "unit": unit,
            "factor": factor,
            "offset": offset
        }

    # 2. Application -> CompuMethod Mapping
    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            app_to_compu[app_name] = compu_ref[0].text.split("/")[-1].strip()

    # 3. Implementation -> BaseType Mapping (DataType Cleanup)
    impl_to_basetype = {}
    for impl_dt in root.xpath("//*[local-name()='IMPLEMENTATION-DATA-TYPE']"):
        impl_name_elem = impl_dt.xpath("*[local-name()='SHORT-NAME']")
        base_ref = impl_dt.xpath(".//*[local-name()='BASE-TYPE-REF']")
        if impl_name_elem and base_ref and base_ref[0].text:
            impl_name = impl_name_elem[0].text.strip()
            basetype_raw = base_ref[0].text.split("/")[-1].strip()
            
            clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', basetype_raw, re.IGNORECASE)
            basetype_name = clean_match.group(1).lower() if clean_match else basetype_raw
            impl_to_basetype[impl_name] = basetype_name

    # 4. Application Records (Ports) and their Elements mapping
    records_dict = {}
    for rec in root.xpath("//*[local-name()='APPLICATION-RECORD-DATA-TYPE']"):
        short_name_elem = rec.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
        rec_name = short_name_elem[0].text.strip()
        
        elements = []
        for elem in rec.xpath(".//*[local-name()='APPLICATION-RECORD-ELEMENT']"):
            elem_name_node = elem.xpath("*[local-name()='SHORT-NAME']")
            tref_node = elem.xpath("*[local-name()='TYPE-TREF']")
            if elem_name_node and tref_node:
                elements.append({
                    "name": elem_name_node[0].text.strip(),
                    "tref": tref_node[0].text.split("/")[-1].strip()
                })
        records_dict[rec_name] = elements

    # 5. NEW: Real Port Extraction from SW-COMPONENT-TYPE
    mapping_to_ports = {}
    for swc in root.xpath("//*[local-name()='APPLICATION-SW-COMPONENT-TYPE']"):
        # Gather all ports in this SWC
        swc_ports = []
        for port in swc.xpath(".//*[local-name()='R-PORT-PROTOTYPE'] | .//*[local-name()='P-PORT-PROTOTYPE']"):
            p_name_node = port.xpath("*[local-name()='SHORT-NAME']")
            if p_name_node:
                swc_ports.append(p_name_node[0].text.strip())

        # Link these ports to their Mapping Sets
        for m_ref in swc.xpath(".//*[local-name()='DATA-TYPE-MAPPING-REF']"):
            if m_ref.text:
                m_short_name = m_ref.text.split("/")[-1].strip()
                if m_short_name not in mapping_to_ports:
                    mapping_to_ports[m_short_name] = set()
                mapping_to_ports[m_short_name].update(swc_ports)

    # Convert sets to lists
    mapping_to_ports = {k: list(v) for k, v in mapping_to_ports.items()}

    # --- Nested Helper Functions ---
    def get_compu_data(app_type_name):
        return compu_methods.get(app_to_compu.get(app_type_name, app_type_name), {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None,
            "unit": "N/A", "factor": "N/A", "offset": "N/A"
        })

    def format_val(v):
        if v is None or v == "N/A": return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    def find_best_port(app_name, event_name, available_ports):
        if not available_ports: return "N/A"
        if len(available_ports) == 1: return available_ports[0]
        # Match against App Name (e.g., 'gadeEvent' in 'SomeIpgadeEventInterface')
        for p in available_ports:
            if app_name.lower() in p.lower(): return p
        # Match against Event Name (e.g., 'GadeSignal' in 'SomeIpgadeEventInterface')
        for p in available_ports:
            if event_name.lower() in p.lower(): return p
        return " | ".join(available_ports)

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
            
        mapping_short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', mapping_short_name)
        if not match: continue
            
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        # Retrieve all real ports associated with this Mapping Set
        swc_ports = mapping_to_ports.get(mapping_short_name, [])
        
        valid_methods = []
        valuestate_app_name = None
        valuestate_impl_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            impl_ref = dt_map.xpath("*[local-name()='IMPLEMENTATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                impl_path_raw = impl_ref[0].text.split("/")[-1].strip() if impl_ref and impl_ref[0].text else None
                
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    valuestate_impl_name = impl_path_raw
                    continue 
                    
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw, impl_path_raw))
        
        for clean_method, raw_app_name, raw_impl_name in valid_methods:
            
            # --- FIND THE REAL PORT ---
            actual_port_name = find_best_port(raw_app_name, raw_event_name, swc_ports)
            
            # --- SCENARIO A: Application Record (Unroll the Port) ---
            if raw_app_name in records_dict:
                for element in records_dict[raw_app_name]:
                    elem_name = element["name"]
                    tref = element["tref"] 
                    
                    c_data = get_compu_data(tref)
                    clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', tref, re.IGNORECASE)
                    datatype = clean_match.group(1).lower() if clean_match else "N/A"
                    
                    states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()]) if c_data["has_enums"] else ("Physical Value" if c_data["min"] is not None else "No Data")
                    
                    sig_str = f'"EthernetCluster::sif_{sif}::{someip_event}::{raw_app_name}::{elem_name}"'
                    parsed_data.append({
                        "Cluster": "EthernetCluster", "SIF": sif, "Event": someip_event,
                        "Port": actual_port_name, "Method": elem_name, "DataType": datatype,
                        "Enums": states_str, "Min": format_val(c_data["min"]),
                        "Mid": format_val(c_data["mid"]), "Max": format_val(c_data["max"]),
                        "Factor": format_val(c_data["factor"]), "Offset": format_val(c_data["offset"]),
                        "Unit": c_data["unit"], "Signal_String": sig_str
                    })

            # --- SCENARIO B: Primitive Value ---
            else:
                c_data = get_compu_data(raw_app_name)
                datatype = impl_to_basetype.get(raw_impl_name, "N/A") if raw_impl_name else "N/A"
                
                states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()]) if c_data["has_enums"] else ("Physical Value" if c_data["min"] is not None else "No Data")

                sig_str = f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"'
                parsed_data.append({
                    "Cluster": "EthernetCluster", "SIF": sif, "Event": someip_event,
                    "Port": actual_port_name, "Method": clean_method, "DataType": datatype,
                    "Enums": states_str, "Min": format_val(c_data["min"]),
                    "Mid": format_val(c_data["mid"]), "Max": format_val(c_data["max"]),
                    "Factor": format_val(c_data["factor"]), "Offset": format_val(c_data["offset"]),
                    "Unit": c_data["unit"], "Signal_String": sig_str
                })
            
            # --- VALUE_STATE EXPANSION ---
            if valuestate_app_name and raw_app_name not in records_dict:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                
                vs_c_data = get_compu_data(valuestate_app_name)
                vs_datatype = impl_to_basetype.get(valuestate_impl_name, "N/A") if valuestate_impl_name else "N/A"
                vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                
                vs_sig_str = f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"'
                parsed_data.append({
                    "Cluster": "EthernetCluster", "SIF": sif, "Event": vs_event,
                    "Port": actual_port_name, # Inherits the port from its parent method
                    "Method": vs_method, "DataType": vs_datatype, "Enums": vs_states,
                    "Min": format_val(vs_c_data["min"]), "Mid": format_val(vs_c_data["mid"]),
                    "Max": format_val(vs_c_data["max"]), "Factor": format_val(vs_c_data["factor"]),
                    "Offset": format_val(vs_c_data["offset"]), "Unit": vs_c_data["unit"],
                    "Signal_String": vs_sig_str
                })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Port', 'Method']).reset_index(drop=True)
    
    return df

# ==========================================
# EXECUTION BLOCK (Run directly in Jupyter)
# ==========================================
file_name = "ETH_CAN.arxml" 
df_signals = parse_arxml_complete(file_name)

if not df_signals.empty:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 40)
    
    # Let's inspect the new accurate Port mappings
    cols_to_view = ['SIF', 'Port', 'Method', 'DataType', 'Enums', 'Min', 'Max']
    display(df_signals[cols_to_view].head(25))

Parsing ETH_CAN.arxml (Extracting Real Ports & Cleaning Types)...


,SIF,Port,Method,DataType,Available_States,Min,Max
0,16414,SomeIpgadeEventInterface,GadeStatus,uint8,0: GADESTATUS_PARK_MODE | 1: GADESTA...,0,7
1,16414,SomeIpgadeEventInterface,GadeStatusValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,2
2,16414,SomeIpgadeEventInterface,gadeStatus,N/A,0: GADESTATUS_PARK_MODE | 1: GADESTA...,0,7
3,16414,SomeIpgadeEventInterface,gadeStatusValueState,N/A,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,2
4,16422,setMinVoltageACInterface,minVoltageValue,N/A,Physical Value,10,16
5,16422,setMinVoltageACInterface,valueState,N/A,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,2
6,16422,setMinVoltageWashingInterface,minVoltageValue,N/A,Physical Value,10,16
7,16422,setMinVoltageWashingInterface,valueState,N/A,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,2
8,16422,setMinVoltageWipingInterface,minVoltageValue,N/A,Physical Value,10,16
9,16422,setMinVoltageWipingInterface,valueState,N/A,0: VALUE_STATE_UNAVAILABLE | 1: VALU...,0,2


In [ ]:
# Filter the dataframe to show only rows where 'Port' contains the pipe character
df_multiple_ports = df_signals[df_signals['Port'].str.contains(r'\|', regex=True, na=False)]

# Get the exact count
print(f"Signals assigned to multiple ports: {len(df_multiple_ports)}")

# View them
display(df_multiple_ports[['Port', 'Method', 'DataType']].head(10))

Signals assigned to multiple ports: 0


,Port,Method,DataType


In [ ]:
import os
import re
import math
import pandas as pd
from lxml import etree
from IPython.display import display

def parse_arxml_complete(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Exploding Shared Ports & Cleaning Types)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING
    # ==========================================
    
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem: continue
        cm_name = cm_name_elem[0].text.strip()
        
        unit_ref = cm.xpath("*[local-name()='UNIT-REF']")
        unit = unit_ref[0].text.split("/")[-1].strip() if unit_ref and unit_ref[0].text else "N/A"
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        factor = "N/A"
        offset = "N/A"
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            coeffs_node = scale.xpath(".//*[local-name()='COMPU-RATIONAL-COEFFS']")
            
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
                
            if coeffs_node:
                try:
                    num_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-NUMERATOR']/*[local-name()='V']")
                    den_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-DENOMINATOR']/*[local-name()='V']")
                    
                    n0 = float(num_v[0].text) if len(num_v) > 0 else 0.0
                    n1 = float(num_v[1].text) if len(num_v) > 1 else 1.0
                    d = float(den_v[0].text) if len(den_v) > 0 else 1.0
                    
                    if d != 0:
                        offset = n0 / d
                        factor = n1 / d
                except Exception: pass
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        mid_val = math.floor((min_val + max_val) / 2) if has_limits else None
            
        compu_methods[cm_name] = {
            "enums": enums,
            "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None,
            "max": max_val if has_limits else None,
            "mid": mid_val,
            "unit": unit,
            "factor": factor,
            "offset": offset
        }

    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            app_to_compu[app_name] = compu_ref[0].text.split("/")[-1].strip()
            

    impl_to_basetype = {}
    for impl_dt in root.xpath("//*[local-name()='IMPLEMENTATION-DATA-TYPE']"):
        impl_name_elem = impl_dt.xpath("*[local-name()='SHORT-NAME']")
        base_ref = impl_dt.xpath(".//*[local-name()='BASE-TYPE-REF']")
        if impl_name_elem and base_ref and base_ref[0].text:
            impl_name = impl_name_elem[0].text.strip()
            basetype_raw = base_ref[0].text.split("/")[-1].strip()
            
            clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', basetype_raw, re.IGNORECASE)
            impl_to_basetype[impl_name] = clean_match.group(1).lower() if clean_match else basetype_raw

    app_to_basetype = {}
    for dt_map in root.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
        a_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
        i_ref = dt_map.xpath("*[local-name()='IMPLEMENTATION-DATA-TYPE-REF']")
        if a_ref and i_ref and a_ref[0].text and i_ref[0].text:
            a_name = a_ref[0].text.split("/")[-1].strip()
            i_name = i_ref[0].text.split("/")[-1].strip()
            basetype = impl_to_basetype.get(i_name, "N/A")
            if basetype != "N/A":
                app_to_basetype[a_name] = basetype

    records_dict = {}
    for rec in root.xpath("//*[local-name()='APPLICATION-RECORD-DATA-TYPE']"):
        short_name_elem = rec.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
        rec_name = short_name_elem[0].text.strip()
        
        elements = []
        for elem in rec.xpath(".//*[local-name()='APPLICATION-RECORD-ELEMENT']"):
            elem_name_node = elem.xpath("*[local-name()='SHORT-NAME']")
            tref_node = elem.xpath("*[local-name()='TYPE-TREF']")
            if elem_name_node and tref_node:
                elements.append({
                    "name": elem_name_node[0].text.strip(),
                    "tref": tref_node[0].text.split("/")[-1].strip()
                })
        records_dict[rec_name] = elements

    mapping_to_ports = {}
    for swc in root.xpath("//*[local-name()='APPLICATION-SW-COMPONENT-TYPE']"):
        swc_ports = []
        for port in swc.xpath(".//*[local-name()='R-PORT-PROTOTYPE'] | .//*[local-name()='P-PORT-PROTOTYPE']"):
            p_name_node = port.xpath("*[local-name()='SHORT-NAME']")
            if p_name_node:
                swc_ports.append(p_name_node[0].text.strip())

        for m_ref in swc.xpath(".//*[local-name()='DATA-TYPE-MAPPING-REF']"):
            if m_ref.text:
                m_short_name = m_ref.text.split("/")[-1].strip()
                if m_short_name not in mapping_to_ports:
                    mapping_to_ports[m_short_name] = set()
                mapping_to_ports[m_short_name].update(swc_ports)

    mapping_to_ports = {k: list(v) for k, v in mapping_to_ports.items()}

    # --- Nested Helper Functions ---
    def get_compu_data(app_type_name):
        return compu_methods.get(app_to_compu.get(app_type_name, app_type_name), {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None,
            "unit": "N/A", "factor": "N/A", "offset": "N/A"
        })

    def format_val(v):
        if v is None or v == "N/A": return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    # UPDATED: Now returns a LIST of ports instead of a pipe-separated string
    def find_best_ports(app_name, event_name, available_ports):
        if not available_ports: return ["N/A"]
        if len(available_ports) == 1: return [available_ports[0]]
        
        # Exact match attempts
        for p in available_ports:
            if app_name.lower() in p.lower(): return [p]
        for p in available_ports:
            if event_name.lower() in p.lower(): return [p]
            
        # If it's a shared primitive, return ALL associated ports to explode the rows
        return available_ports 

    # ==========================================
    # MAIN PARSING
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
            
        mapping_short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', mapping_short_name)
        if not match: continue
            
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        swc_ports = mapping_to_ports.get(mapping_short_name, [])
        
        valid_methods = []
        valuestate_app_name = None
        valuestate_impl_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            impl_ref = dt_map.xpath("*[local-name()='IMPLEMENTATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                impl_path_raw = impl_ref[0].text.split("/")[-1].strip() if impl_ref and impl_ref[0].text else None
                
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    valuestate_impl_name = impl_path_raw
                    continue 
                    
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw, impl_path_raw))
        
        for clean_method, raw_app_name, raw_impl_name in valid_methods:
            
            # --- THE FIX: Retrieve as a List and iterate ---
            actual_ports = find_best_ports(raw_app_name, raw_event_name, swc_ports)
            
            for actual_port_name in actual_ports:
            
                if raw_app_name in records_dict:
                    for element in records_dict[raw_app_name]:
                        elem_name = element["name"]
                        tref = element["tref"] 
                        
                        c_data = get_compu_data(tref)
                        
                        # --- THE FIX ---
                        # First, try to look up the exact base type using our new translation map
                        datatype = app_to_basetype.get(tref, "N/A")
                        
                        # Fallback to regex if the map didn't catch it
                        if datatype == "N/A":
                            clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', tref, re.IGNORECASE)
                            datatype = clean_match.group(1).lower() if clean_match else "N/A"
                        # ---------------
                        
                        states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()]) if c_data["has_enums"] else ("Physical Value" if c_data["min"] is not None else "No Data")
                        
                        sig_str = f'"EthernetCluster::sif_{sif}::{someip_event}::{raw_app_name}::{elem_name}"'
                        parsed_data.append({
                            "Cluster": "EthernetCluster", "SIF": sif, "Event": someip_event,
                            "Port": actual_port_name, "Method": elem_name, "DataType": datatype,
                            "Enums": states_str, "Min": format_val(c_data["min"]),
                            "Mid": format_val(c_data["mid"]), "Max": format_val(c_data["max"]),
                            "Factor": format_val(c_data["factor"]), "Offset": format_val(c_data["offset"]),
                            "Unit": c_data["unit"], "Signal_String": sig_str
                        })

                else:
                    c_data = get_compu_data(raw_app_name)
                    datatype = impl_to_basetype.get(raw_impl_name, "N/A") if raw_impl_name else "N/A"
                    
                    states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()]) if c_data["has_enums"] else ("Physical Value" if c_data["min"] is not None else "No Data")

                    sig_str = f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"'
                    parsed_data.append({
                        "Cluster": "EthernetCluster", "SIF": sif, "Event": someip_event,
                        "Port": actual_port_name, "Method": clean_method, "DataType": datatype,
                        "Enums": states_str, "Min": format_val(c_data["min"]),
                        "Mid": format_val(c_data["mid"]), "Max": format_val(c_data["max"]),
                        "Factor": format_val(c_data["factor"]), "Offset": format_val(c_data["offset"]),
                        "Unit": c_data["unit"], "Signal_String": sig_str
                    })
                
                # VALUE_STATE EXPANSION
                if valuestate_app_name and raw_app_name not in records_dict:
                    vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                    vs_method = f"{clean_method}ValueState"
                    
                    vs_c_data = get_compu_data(valuestate_app_name)
                    vs_datatype = impl_to_basetype.get(valuestate_impl_name, "N/A") if valuestate_impl_name else "N/A"
                    vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                    
                    vs_sig_str = f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"'
                    parsed_data.append({
                        "Cluster": "EthernetCluster", "SIF": sif, "Event": vs_event,
                        "Port": actual_port_name, "Method": vs_method, "DataType": vs_datatype, 
                        "Enums": vs_states, "Min": format_val(vs_c_data["min"]), 
                        "Mid": format_val(vs_c_data["mid"]), "Max": format_val(vs_c_data["max"]), 
                        "Factor": format_val(vs_c_data["factor"]), "Offset": format_val(vs_c_data["offset"]), 
                        "Unit": vs_c_data["unit"], "Signal_String": vs_sig_str
                    })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Port', 'Method']).reset_index(drop=True)
    
    return df

In [ ]:
file_name = "ETH_CAN.arxml" 
df_signals = parse_arxml_complete(file_name)

if not df_signals.empty:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    
    # Let's inspect the new accurate Port mappings
    cols_to_view = ['SIF', 'Port', 'Method', 'DataType', 'Enums', 'Min', 'Max']
    display(df_signals[cols_to_view].head(25))

Parsing ETH_CAN.arxml (Exploding Shared Ports & Cleaning Types)...


,SIF,Port,Method,DataType,Available_States,Min,Max
0,16414,SomeIpgadeEventInterface,GadeStatus,uint8,0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_ON_BOARD | 2: GADESTATUS_MISSION_MODE_LOW_BATT | 3: GADESTATUS_MISSION_MODE_ON | 4: GADESTATUS_SW_UPDATE_MODE | 7: GADESTATUS_UNAVAILABLE,0,7
1,16414,SomeIpgadeEventInterface,GadeStatusValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: VALUE_STATE_INVALID,0,2
2,16414,SomeIpgadeEventInterface,gadeStatus,uint8,0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_ON_BOARD | 2: GADESTATUS_MISSION_MODE_LOW_BATT | 3: GADESTATUS_MISSION_MODE_ON | 4: GADESTATUS_SW_UPDATE_MODE | 7: GADESTATUS_UNAVAILABLE,0,7
3,16414,SomeIpgadeEventInterface,gadeStatusValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: VALUE_STATE_INVALID,0,2
4,16422,setMinVoltageACInterface,VoltageValueType,float32,Physical Value,10,16
5,16422,setMinVoltageACInterface,VoltageValueTypeValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: VALUE_STATE_INVALID,0,2
6,16422,setMinVoltageACInterface,minVoltageValue,float32,Physical Value,10,16
7,16422,setMinVoltageACInterface,minVoltageValue,float32,Physical Value,10,16
8,16422,setMinVoltageACInterface,minVoltageValue1,float32,Physical Value,10,16
9,16422,setMinVoltageACInterface,minVoltageValue1ValueState,uint8,0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: VALUE_STATE_INVALID,0,2


In [ ]:
import pandas as pd
from IPython.display import display

def validate_requirements_relaxed(req_csv_path, df_signals):
    print(f"Loading Requirements from: {req_csv_path}...")
    
    try:
        df_req = pd.read_csv(req_csv_path)
    except FileNotFoundError:
        print(f"Error: Could not find '{req_csv_path}'")
        return None, None

    # ==========================================
    # 1. NORMALIZE KEYS (SIF & Method ONLY)
    # ==========================================
    # We drop 'Port' from the matching criteria to bypass the naming convention mismatch
    
    df_req['match_sif'] = df_req['Service ID'].astype(str).str.strip().str.replace('.0', '', regex=False)
    df_req['match_method'] = df_req['Attribute value'].astype(str).str.strip().str.lower()
    
    df_signals['match_sif'] = df_signals['SIF'].astype(str).str.strip()
    df_signals['match_method'] = df_signals['Method'].astype(str).str.strip().str.lower()

    # Drop duplicate methods in df_signals just for the merge so we don't multiply rows
    df_signals_unique = df_signals.drop_duplicates(subset=['match_sif', 'match_method'])

    # ==========================================
    # 2. PERFORM THE RELAXED MATCH
    # ==========================================
    merged_df = pd.merge(
        df_req, 
        df_signals_unique[['match_sif', 'match_method', 'Signal_String', 'DataType', 'Enums', 'Min', 'Max', 'Event', 'Port']], 
        on=['match_sif', 'match_method'], 
        how='left',
        suffixes=('_CSV', '_ARXML') # In case of column name overlap
    )

    merged_df['Is_Found_In_DB'] = merged_df['Signal_String'].notna()

    # ==========================================
    # 3. STATISTICS
    # ==========================================
    total_reqs = len(merged_df)
    matched_reqs = merged_df['Is_Found_In_DB'].sum()
    missing_reqs = total_reqs - matched_reqs
    match_percentage = (matched_reqs / total_reqs) * 100 if total_reqs > 0 else 0

    print("=== RELAXED VALIDATION STATISTICS ===")
    print(f"Total Requirements   : {total_reqs}")
    print(f"✅ Matched (SIF+Method): {matched_reqs} ({match_percentage:.2f}%)")
    print(f"❌ Still Missing       : {missing_reqs}\n")

    df_matched = merged_df[merged_df['Is_Found_In_DB'] == True].copy()
    df_missing = merged_df[merged_df['Is_Found_In_DB'] == False].copy()

    # Cleanup temp columns
    df_matched = df_matched.drop(columns=['match_sif', 'match_method', 'Is_Found_In_DB'])
    df_missing = df_missing.drop(columns=['match_sif', 'match_method', 'Is_Found_In_DB'])

    return df_matched, df_missing

# ==========================================
# EXECUTION
# ==========================================
req_file_path = "test_requirements_ E2E_ETH.csv"

df_matched, df_missing = validate_requirements_relaxed(req_file_path, df_signals)

if df_matched is not None and not df_matched.empty:
    print("--- 🟢 Top 10 SUCCESSFULLY MATCHED Requirements ---")
    pd.set_option('display.max_columns', None)
    
    # We display the CSV's requested Port vs the ARXML's actual Port
    display_cols_matched = [
        'E2E REQ (used for implementation)', 'Service ID', 'Attribute value', 
        'Port_CSV', 'Port_ARXML', 'Signal_String', 'DataType'
    ]
    # If the CSV column is named 'Port', the merge suffix changed it to 'Port_CSV'
    display(df_matched[[c for c in display_cols_matched if c in df_matched.columns]].head(10))

if df_missing is not None and not df_missing.empty:
    print("\n--- 🔴 Top 10 STILL MISSING Requirements ---")
    display_cols_missing = ['E2E REQ (used for implementation)', 'Service ID', 'Attribute value', 'Port']
    # Select columns safely
    missing_cols_to_show = [c for c in display_cols_missing if c in df_missing.columns]
    if 'Port_CSV' in df_missing.columns and 'Port' not in df_missing.columns:
        missing_cols_to_show.append('Port_CSV')
        
    display(df_missing[missing_cols_to_show].head(10))

Loading Requirements from: test_requirements_ E2E_ETH.csv...
=== RELAXED VALIDATION STATISTICS ===
Total Requirements   : 2305
✅ Matched (SIF+Method): 2302 (99.87%)
❌ Still Missing       : 3

--- 🟢 Top 10 SUCCESSFULLY MATCHED Requirements ---


,E2E REQ (used for implementation),Service ID,Attribute value,Port_CSV,Port_ARXML,Signal_String,DataType
0,REQ_ETH_E2E_20_1_1,591,absInRegulation,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationStates::AbsInRegulation""",uint8
1,REQ_ETH_E2E_20_2_1,591,absInRegulation,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationStates::AbsInRegulation""",uint8
2,REQ_ETH_E2E_20_3_1,591,absInRegulation,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationStates::AbsInRegulation""",uint8
3,REQ_ETH_E2E_20_1_0,591,absInRegulationValueState,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationState::AbsInRegulationValueState""",uint8
4,REQ_ETH_E2E_20_2_0,591,absInRegulationValueState,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationState::AbsInRegulationValueState""",uint8
5,REQ_ETH_E2E_20_3_0,591,absInRegulationValueState,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationState::AbsInRegulationValueState""",uint8
6,REQ_ETH_E2E_21_1_3,591,absMalfunction,ChassisBrakingRegulationStates_ChassisRegulationMalfunctionState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationStates::AbsMalfunction""",uint8
7,REQ_ETH_E2E_21_1_2,591,absMalfunctionValueState,ChassisBrakingRegulationStates_ChassisRegulationMalfunctionState,SomeIpChassisRegulationActivationStateInterface,"""EthernetCluster::sif_591::SomeIpChassisBrakingRegulationState::AbsMalfunctionValueState""",uint8
8,REQ_ETH_E2E_24_0_1,645,absStateDisplayRequest,ChassisAlerts_BrakeAndSteeringAlerts,SomeIpBrakeAndSteeringAlertsInterface,"""EthernetCluster::sif_645::SomeIpChassisAlerts::AbsStateDisplayRequest""",uint8
9,REQ_ETH_E2E_24_0_2,645,absStateDisplayRequestOccurence,ChassisAlerts_BrakeAndSteeringAlerts,SomeIpBrakeAndSteeringAlertsInterface,"""EthernetCluster::sif_645::SomeIpChassisAlerts::BrakeAndSteeringAlerts::absStateDisplayRequestOccurence""",uint8



--- 🔴 Top 10 STILL MISSING Requirements ---


,E2E REQ (used for implementation),Service ID,Attribute value,Port_CSV
1033,REQ_ETH_E2E_325_1_1,16438,isVehicleUtcRequested,RequestVehicleUtcTime_requestVehicleUtcTime
1405,REQ_ETH_E2E_325_1_0,16438,requestVehicleUtcValueState,RequestVehicleUtcTime_requestVehicleUtcTime
1758,REQ_ETH_E2E_419_1_0,sdv.body.climate.cabin.thermal.HeatLoopWaterStatus,valueState,ThermoMgt_HeatLoopWaterTempV2Event


In [ ]:
import pandas as pd
from IPython.display import display

# ==========================================
# 1. FIX PANDAS TRUNCATION
# ==========================================
# Setting max_colwidth to None forces Jupyter to show the entire string, no matter how long.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 2000)

# ==========================================
# 2. INVESTIGATE SIF 16438 (Vehicle UTC Time)
# ==========================================
print("=== ARXML DATABASE: SIF 16438 ===")
# Filter the database to show everything under SIF 16438
df_utc = df_signals[df_signals['SIF'] == '16438']

if not df_utc.empty:
    display(df_utc[['SIF', 'Event', 'Port', 'Method', 'DataType', 'Signal_String']])
else:
    print("No signals found for SIF 16438. The OEM may have forgotten to include this Service in the ARXML export.")

print("\n" + "="*80 + "\n")

# ==========================================
# 3. INVESTIGATE THE CORRUPTED REQUIREMENT
# ==========================================
print("=== ARXML DATABASE: HeatLoopWaterTempV2 ===")
# Since the SIF is corrupted in the CSV, we will search the ARXML by the Port/Method name
df_heat = df_signals[
    df_signals['Port'].str.contains('HeatLoopWaterTemp', case=False, na=False) |
    df_signals['Event'].str.contains('HeatLoopWaterTemp', case=False, na=False) |
    df_signals['Method'].str.contains('HeatLoopWaterTemp', case=False, na=False)
]

if not df_heat.empty:
    display(df_heat[['SIF', 'Event', 'Port', 'Method', 'DataType', 'Signal_String']])
else:
    print("No signals found matching 'HeatLoopWaterTemp'.")

=== ARXML DATABASE: SIF 16438 ===


,SIF,Event,Port,Method,DataType,Signal_String
371,16438,SomeIpRequestVehicleUtcTime,requestVehicleUtcTimeInterface,requestVehicleUtcTimeValueState,uint8,"""EthernetCluster::sif_16438::SomeIpRequestVehicleUtcTime::requestVehicleUtcTime::requestVehicleUtcTimeValueState"""




=== ARXML DATABASE: HeatLoopWaterTempV2 ===


,SIF,Event,Port,Method,DataType,Signal_String
35965,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,CabinHeatingRequest,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::CabinHeatingRequest"""
35966,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,CabinHeatingRequestValueState,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::CabinHeatingRequestValueState"""
35967,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,ClimLoopMode,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::ClimLoopMode"""
35968,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,ClimLoopModeValueState,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::ClimLoopModeValueState"""
35969,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,EvaporatorFlagRequest,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::EvaporatorFlagRequest"""
35970,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,EvaporatorFlagRequestValueState,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::EvaporatorFlagRequestValueState"""
35971,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,HVBattCondMode,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::HVBattCondMode"""
35972,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,HVBattCondModeValueState,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::HVBattCondModeValueState"""
35973,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,TractionBattCooling,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::TractionBattCooling"""
35974,8312,SomeIpThermoMgt,SomeIpHeatLoopWaterTempV2EventInterface,TractionBattCoolingValueState,uint8,"""EthernetCluster::sif_8312::SomeIpThermoMgt::TractionBattCoolingValueState"""


In [ ]:
from lxml import etree

tree = etree.parse("ETH_CAN.arxml")
root = tree.getroot()

# Search for the record by its short name
target_name = "requestVehicleUtcTime"
found_nodes = root.xpath(f"//*[*[local-name()='SHORT-NAME']='{target_name}']")

print(f"Found {len(found_nodes)} matching nodes for '{target_name}'.\n")

for node in found_nodes:
    tag_name = etree.QName(node).localname
    print(f"--- XML TAG: <{tag_name}> ---")
    # Print the first 15 lines of the XML block to see its structure
    xml_str = etree.tostring(node, pretty_print=True).decode('utf-8')
    print("\n".join(xml_str.splitlines()[:15]))
    print("...\n")

Found 4 matching nodes for 'requestVehicleUtcTime'.

--- XML TAG: <APPLICATION-RECORD-DATA-TYPE> ---
<APPLICATION-RECORD-DATA-TYPE xmlns="http://autosar.org/schema/r4.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" UUID="Obfa81d0b195123df24866a5fXObfa81d0b195123df24866a5900">
              <SHORT-NAME>requestVehicleUtcTime</SHORT-NAME>
              <CATEGORY>STRUCTURE</CATEGORY>
              <ELEMENTS>
                <APPLICATION-RECORD-ELEMENT UUID="Obfa81d0b195123df24866a79XObfa81d0b195123df24866a7300">
                  <SHORT-NAME>requestVehicleUtcTimeValueState</SHORT-NAME>
                  <CATEGORY>VALUE</CATEGORY>
                  <TYPE-TREF DEST="APPLICATION-PRIMITIVE-DATA-TYPE">/DataTypes/ApplicationDataTypes/ValueState86</TYPE-TREF>
                </APPLICATION-RECORD-ELEMENT>
              </ELEMENTS>
            </APPLICATION-RECORD-DATA-TYPE>
            
...

--- XML TAG: <IMPLEMENTATION-DATA-TYPE> ---
<IMPLEMENTATION-DATA-TYPE xmlns="http://autosar.org/sc

In [ ]:
from lxml import etree

tree = etree.parse("ETH_CAN.arxml")
root = tree.getroot()

search_term = "isVehicleUtcRequested"

# Search for ANY tag that has a SHORT-NAME containing our search term
found_nodes = root.xpath(f"//*[contains(translate(*[local-name()='SHORT-NAME'], 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{search_term.lower()}')]")

if not found_nodes:
    print(f"❌ DEFINITIVE PROOF: The exact string '{search_term}' literally does NOT exist anywhere in this ARXML file as a SHORT-NAME.")
else:
    print(f"Found {len(found_nodes)} matches! They are hiding here:")
    for node in found_nodes:
        tag_name = etree.QName(node).localname
        name = node.xpath("*[local-name()='SHORT-NAME']")[0].text
        print(f"Tag: <{tag_name}> | Name: {name}")

❌ DEFINITIVE PROOF: The exact string 'isVehicleUtcRequested' literally does NOT exist anywhere in this ARXML file as a SHORT-NAME.


In [20]:
# Search for 'HeatLoopWaterTemp' across Event, Port, and Method columns (case-insensitive)
# search_term = 'HeatLoopWaterTempV2'
# search_term = 'AclowpressureEvent'
search_term = 'climloopmodeevent'

df_heat = df_signals[
    df_signals['Event'].str.contains(search_term, case=False, na=False) |
    df_signals['Port'].str.contains(search_term, case=False, na=False) |
    df_signals['Method'].str.contains(search_term, case=False, na=False)
]

if not df_heat.empty:
    print(f"✅ Found {len(df_heat)} matching signals!")
    
    # Configure view to prevent truncation
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    
    # Display the core attributes so we can figure out the real SIF and Method name
    display(df_heat[['SIF', 'Event', 'Port', 'Method', 'DataType', 'Signal_String']])
else:
    print(f"❌ '{search_term}' does not exist anywhere in the parsed ARXML data.")

✅ Found 20 matching signals!


,SIF,Event,Port,Method,DataType,Signal_String
33349,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,CabinHeatingRequest,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::CabinHeatingRequest"""
33350,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,CabinHeatingRequestValueState,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::CabinHeatingRequestValueState"""
33351,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,ClimLoopMode,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::ClimLoopMode"""
33352,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,ClimLoopModeValueState,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::ClimLoopModeValueState"""
33353,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,EvaporatorFlagRequest,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::EvaporatorFlagRequest"""
33354,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,EvaporatorFlagRequestValueState,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::EvaporatorFlagRequestValueState"""
33355,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,HVBattCondMode,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::HVBattCondMode"""
33356,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,HVBattCondModeValueState,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::HVBattCondModeValueState"""
33357,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,TractionBattCooling,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::TractionBattCooling"""
33358,8312,SomeIpThermoMgt,SomeIpClimLoopModeEventInterface,TractionBattCoolingValueState,uint8,"""EthernetCluster::sif_8312::SomeIpClimLoopModeEvent::TractionBattCoolingValueState"""


In [ ]:
import pandas as pd
from IPython.display import display

print("="*50)
print(" 🚀 ARXML PARSING HEALTH CHECK DASHBOARD")
print("="*50)

# 1. Basic Shape and Duplicates
total_signals = len(df_signals)
duplicate_signals = df_signals.duplicated(subset=['Signal_String']).sum()

print(f"Total Signals Extracted : {total_signals}")
print(f"Duplicate Signal Strings: {duplicate_signals} ", end="")
print("(🚨 WARNING!)" if duplicate_signals > 0 else "(✅ Clean)")

print("\n" + "="*50)
print(" 📊 COLUMN-LEVEL 'N/A' COUNTS")
print("="*50)
# Count literal "N/A" strings across all columns
na_counts = (df_signals == "N/A").sum()
# Only show columns that actually have "N/A" values
display(na_counts[na_counts > 0].to_frame(name="Missing ('N/A') Count"))

print("\n" + "="*50)
print(" 🧩 DATA TYPE DISTRIBUTION")
print("="*50)
# This will show you if any 'N/A' DataTypes slipped through and what the most common types are
display(df_signals['DataType'].value_counts().to_frame(name="Count"))

print("\n" + "="*50)
print(" 📏 PHYSICAL vs. ENUM BREAKDOWN")
print("="*50)
physical_count = (df_signals['Enums'] == "Physical Value").sum()
no_data_count = (df_signals['Enums'] == "No Data").sum()
enum_count = total_signals - physical_count - no_data_count

print(f"Discrete Enum Signals : {enum_count}")
print(f"Continuous Physical   : {physical_count}")
print(f"Missing State Config  : {no_data_count}")

print("\n" + "="*50)
print(" ⚖️ UNIT USAGE (Top 10)")
print("="*50)
# Filter out "N/A" to see which physical units are most heavily used
units_used = df_signals[df_signals['Unit'] != "N/A"]['Unit'].value_counts()
if not units_used.empty:
    display(units_used.head(10).to_frame(name="Count"))
else:
    print("No units defined in this database.")

# ==========================================
# DEEP DIVE: View the Problematic Rows
# ==========================================
# If you saw "N/A" in the DataType check above, uncomment this to see exactly which ones they are!

# print("\n--- 🔍 Investigating 'N/A' DataTypes ---")
# df_bad_types = df_signals[df_signals['DataType'] == "N/A"]
# if not df_bad_types.empty:
#     display(df_bad_types[['SIF', 'Port', 'Method', 'Signal_String']].head(15))

 🚀 ARXML PARSING HEALTH CHECK DASHBOARD
Total Signals Extracted : 44397
Duplicate Signal Strings: 0 (✅ Clean)

 📊 COLUMN-LEVEL 'N/A' COUNTS


,Missing ('N/A') Count
DataType,279
Min,1213
Mid,1213
Max,1213
Factor,34599
Offset,34599
Unit,37748



 🧩 DATA TYPE DISTRIBUTION


,Count
DataType,
uint8,35748
float32,4234
uint32,1550
uint16,891
boolean,710
sint16,282
N/A,279
uint64,275
sint32,214



 📏 PHYSICAL vs. ENUM BREAKDOWN
Discrete Enum Signals : 33442
Continuous Physical   : 9742
Missing State Config  : 1213

 ⚖️ UNIT USAGE (Top 10)


,Count
Unit,
CELSIUS,888
PERCENTAGE,447
KW,420
V1,339
MINUTE,240
X__C1,238
AMPERE,235
A1,213
X_3,213


In [25]:
import pandas as pd
from IPython.display import display

# Configure view
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("=== 🚨 INVESTIGATING THE DUPLICATES ===")
# Grab all rows that share a Signal_String with another row
df_dups = df_signals[df_signals.duplicated(subset=['Signal_String'], keep=False)]

# Sort them so the duplicates are stacked right next to each other
display(df_dups.sort_values('Signal_String')[['SIF', 'Port', 'Event', 'Method', 'Signal_String']].head(20))

print("\n" + "="*80 + "\n")

print("=== 🧩 INVESTIGATING THE 444 'N/A' DATA TYPES ===")
df_na_types = df_signals[df_signals['DataType'] == "N/A"]
display(df_na_types[['SIF', 'Port', 'Method', 'Signal_String']].head(20))

=== 🚨 INVESTIGATING THE DUPLICATES ===


,SIF,Port,Event,Method,Signal_String




=== 🧩 INVESTIGATING THE 444 'N/A' DATA TYPES ===


,SIF,Port,Method,Signal_String
425,16459,CollectReferencesInterface,errorDetail8,"""EthernetCluster::sif_16459::CollectReferences::errorDetail8"""
430,16459,CollectReferencesInterface,localInventoryFilePathname,"""EthernetCluster::sif_16459::CollectReferences::localInventoryFilePathname"""
438,16459,CollectReferencesInterface,propertiesList,"""EthernetCluster::sif_16459::CollectReferences::propertiesList"""
440,16459,CollectReferencesInterface,refKeysList,"""EthernetCluster::sif_16459::CollectReferences::refKeysList"""
455,16459,SomeIpReferencesCollectionResultInterface,errorDetail,"""EthernetCluster::sif_16459::SomeIpReferencesCollectionResult::errorDetail"""
456,16459,SomeIpReferencesCollectionResultInterface,errorDetail8,"""EthernetCluster::sif_16459::SomeIpReferencesCollectionResult::errorDetail8"""
463,16459,SomeIpReferencesCollectionResultInterface,localInventoryFilePathname,"""EthernetCluster::sif_16459::SomeIpReferencesCollectionResult::localInventoryFilePathname"""
471,16459,SomeIpReferencesCollectionResultInterface,propertiesList,"""EthernetCluster::sif_16459::SomeIpReferencesCollectionResult::propertiesList"""
473,16459,SomeIpReferencesCollectionResultInterface,refKeysList,"""EthernetCluster::sif_16459::SomeIpReferencesCollectionResult::refKeysList"""
487,16459,StopCollectingReferencesInterface,errorDetail8,"""EthernetCluster::sif_16459::StopCollectingReferences::errorDetail8"""


In [26]:
import pandas as pd
from IPython.display import display

# ==========================================
# 1. PORT SUFFIX STATISTICS
# ==========================================
# Isolate all unique ports across the entire database
unique_ports = df_signals['Port'].dropna().unique()
total_ports = len(unique_ports)

# Categorize them
ports_with_interface = [p for p in unique_ports if p.endswith('Interface')]
ports_without_interface = [p for p in unique_ports if not p.endswith('Interface')]

print("="*50)
print(" 🚪 PORT SUFFIX STATISTICS")
print("="*50)
print(f"Total Unique Ports            : {total_ports}")
print(f"Ports ending with 'Interface' : {len(ports_with_interface)}")
print(f"Ports without 'Interface'     : {len(ports_without_interface)}")

# ==========================================
# 2. INVESTIGATE EDGE CASES
# ==========================================
print("\n" + "="*50)
print(" 🔍 PORTS WITHOUT 'Interface' SUFFIX")
print("="*50)
if ports_without_interface:
    print("These ports do not end with 'Interface'. We need to see if they need special handling:")
    for p in ports_without_interface[:20]: # Show first 20 anomalies
        print(f"  - {p}")
else:
    print("✅ All ports cleanly end with 'Interface'! It is 100% safe to strip.")

# ==========================================
# 3. MOCKUP THE NEW SIGNAL STRING
# ==========================================
print("\n" + "="*50)
print(" ✨ PREVIEW OF PROPOSED SIGNAL STRING")
print("="*50)

# Create a temporary copy to test our logic
df_preview = df_signals.copy()

# Fix 1: Strip 'Interface' from the end of the port name
df_preview['Clean_Port'] = df_preview['Port'].str.replace(r'Interface$', '', regex=True)

# Fix 2: Construct the new Signal_String using the Clean_Port
df_preview['New_Signal_String'] = '"EthernetCluster::sif_' + df_preview['SIF'].astype(str) + '::' + df_preview['Clean_Port'] + '::' + df_preview['Method'] + '"'

# Configure view
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Look specifically at SIF 8312 to verify your ThermoMgt theory
sample_sif_8312 = df_preview[df_preview['SIF'] == '8312'].head(10)

if not sample_sif_8312.empty:
    display(sample_sif_8312[['Port', 'Method', 'Signal_String', 'New_Signal_String']])
else:
    display(df_preview[['Port', 'Method', 'Signal_String', 'New_Signal_String']].head(10))

 🚪 PORT SUFFIX STATISTICS
Total Unique Ports            : 1045
Ports ending with 'Interface' : 1045
Ports without 'Interface'     : 0

 🔍 PORTS WITHOUT 'Interface' SUFFIX
✅ All ports cleanly end with 'Interface'! It is 100% safe to strip.

 ✨ PREVIEW OF PROPOSED SIGNAL STRING


,Port,Method,Signal_String,New_Signal_String
33306,SomeIpACLowPressureEventInterface,CabinHeatingRequest,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::CabinHeatingRequest""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::CabinHeatingRequest"""
33307,SomeIpACLowPressureEventInterface,CabinHeatingRequestValueState,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::CabinHeatingRequestValueState""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::CabinHeatingRequestValueState"""
33308,SomeIpACLowPressureEventInterface,ClimLoopMode,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::ClimLoopMode""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::ClimLoopMode"""
33309,SomeIpACLowPressureEventInterface,ClimLoopModeValueState,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::ClimLoopModeValueState""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::ClimLoopModeValueState"""
33310,SomeIpACLowPressureEventInterface,EvaporatorFlagRequest,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::EvaporatorFlagRequest""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::EvaporatorFlagRequest"""
33311,SomeIpACLowPressureEventInterface,EvaporatorFlagRequestValueState,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::EvaporatorFlagRequestValueState""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::EvaporatorFlagRequestValueState"""
33312,SomeIpACLowPressureEventInterface,HVBattCondMode,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::HVBattCondMode""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::HVBattCondMode"""
33313,SomeIpACLowPressureEventInterface,HVBattCondModeValueState,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::HVBattCondModeValueState""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::HVBattCondModeValueState"""
33314,SomeIpACLowPressureEventInterface,TractionBattCooling,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::TractionBattCooling""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::TractionBattCooling"""
33315,SomeIpACLowPressureEventInterface,TractionBattCoolingValueState,"""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::TractionBattCoolingValueState""","""EthernetCluster::sif_8312::SomeIpACLowPressureEvent::TractionBattCoolingValueState"""


somewhere near the end i suppose

In [ ]:
import os
import re
import math
import pandas as pd
from lxml import etree
from IPython.display import display

def parse_arxml_complete(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (Final Architecture: Exploding Ports & Cleaning Paths)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING
    # ==========================================
    
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem: continue
        cm_name = cm_name_elem[0].text.strip()
        
        unit_ref = cm.xpath("*[local-name()='UNIT-REF']")
        unit = unit_ref[0].text.split("/")[-1].strip() if unit_ref and unit_ref[0].text else "N/A"
        
        enums = {}
        min_val = float('inf')
        max_val = float('-inf')
        factor = "N/A"
        offset = "N/A"
        
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            ll_node = scale.xpath("*[local-name()='LOWER-LIMIT']")
            ul_node = scale.xpath("*[local-name()='UPPER-LIMIT']")
            vt_node = scale.xpath(".//*[local-name()='VT']")
            coeffs_node = scale.xpath(".//*[local-name()='COMPU-RATIONAL-COEFFS']")
            
            ll_text = ll_node[0].text.strip() if ll_node and ll_node[0].text else None
            ul_text = ul_node[0].text.strip() if ul_node and ul_node[0].text else ll_text
            
            if ll_text is not None:
                try:
                    ll_f = float(ll_text)
                    if ll_f < min_val: min_val = ll_f
                except ValueError: pass
            
            if ul_text is not None:
                try:
                    ul_f = float(ul_text)
                    if ul_f > max_val: max_val = ul_f
                except ValueError: pass
            
            if ll_text is not None and vt_node and vt_node[0].text:
                enums[ll_text] = vt_node[0].text.strip()
                
            if coeffs_node:
                try:
                    num_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-NUMERATOR']/*[local-name()='V']")
                    den_v = coeffs_node[0].xpath(".//*[local-name()='COMPU-DENOMINATOR']/*[local-name()='V']")
                    
                    n0 = float(num_v[0].text) if len(num_v) > 0 else 0.0
                    n1 = float(num_v[1].text) if len(num_v) > 1 else 1.0
                    d = float(den_v[0].text) if len(den_v) > 0 else 1.0
                    
                    if d != 0:
                        offset = n0 / d
                        factor = n1 / d
                except Exception: pass
        
        has_limits = min_val != float('inf') and max_val != float('-inf')
        mid_val = math.floor((min_val + max_val) / 2) if has_limits else None
            
        compu_methods[cm_name] = {
            "enums": enums, "has_enums": len(enums) > 0,
            "min": min_val if has_limits else None, "max": max_val if has_limits else None,
            "mid": mid_val, "unit": unit, "factor": factor, "offset": offset
        }

    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            app_to_compu[app_name] = compu_ref[0].text.split("/")[-1].strip()

    impl_to_basetype = {}
    for impl_dt in root.xpath("//*[local-name()='IMPLEMENTATION-DATA-TYPE']"):
        impl_name_elem = impl_dt.xpath("*[local-name()='SHORT-NAME']")
        base_ref = impl_dt.xpath(".//*[local-name()='BASE-TYPE-REF']")
        if impl_name_elem and base_ref and base_ref[0].text:
            impl_name = impl_name_elem[0].text.strip()
            basetype_raw = base_ref[0].text.split("/")[-1].strip()
            
            clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', basetype_raw, re.IGNORECASE)
            impl_to_basetype[impl_name] = clean_match.group(1).lower() if clean_match else basetype_raw

    # Direct App->BaseType map (Fixes "N/A" DataType bug for records)
    app_to_basetype = {}
    for dt_map in root.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
        a_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
        i_ref = dt_map.xpath("*[local-name()='IMPLEMENTATION-DATA-TYPE-REF']")
        if a_ref and i_ref and a_ref[0].text and i_ref[0].text:
            a_name = a_ref[0].text.split("/")[-1].strip()
            i_name = i_ref[0].text.split("/")[-1].strip()
            basetype = impl_to_basetype.get(i_name, "N/A")
            if basetype != "N/A":
                app_to_basetype[a_name] = basetype

    records_dict = {}
    for rec in root.xpath("//*[local-name()='APPLICATION-RECORD-DATA-TYPE']"):
        short_name_elem = rec.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
        rec_name = short_name_elem[0].text.strip()
        
        elements = []
        for elem in rec.xpath(".//*[local-name()='APPLICATION-RECORD-ELEMENT']"):
            elem_name_node = elem.xpath("*[local-name()='SHORT-NAME']")
            tref_node = elem.xpath("*[local-name()='TYPE-TREF']")
            if elem_name_node and tref_node:
                elements.append({
                    "name": elem_name_node[0].text.strip(),
                    "tref": tref_node[0].text.split("/")[-1].strip()
                })
        records_dict[rec_name] = elements

    mapping_to_ports = {}
    for swc in root.xpath("//*[local-name()='APPLICATION-SW-COMPONENT-TYPE']"):
        swc_ports = []
        for port in swc.xpath(".//*[local-name()='R-PORT-PROTOTYPE'] | .//*[local-name()='P-PORT-PROTOTYPE']"):
            p_name_node = port.xpath("*[local-name()='SHORT-NAME']")
            if p_name_node:
                swc_ports.append(p_name_node[0].text.strip())

        for m_ref in swc.xpath(".//*[local-name()='DATA-TYPE-MAPPING-REF']"):
            if m_ref.text:
                m_short_name = m_ref.text.split("/")[-1].strip()
                if m_short_name not in mapping_to_ports:
                    mapping_to_ports[m_short_name] = set()
                mapping_to_ports[m_short_name].update(swc_ports)

    mapping_to_ports = {k: list(v) for k, v in mapping_to_ports.items()}

    # --- Nested Helper Functions ---
    def get_compu_data(app_type_name):
        return compu_methods.get(app_to_compu.get(app_type_name, app_type_name), {
            "enums": {}, "has_enums": False, 
            "min": None, "max": None, "mid": None,
            "unit": "N/A", "factor": "N/A", "offset": "N/A"
        })

    def format_val(v):
        if v is None or v == "N/A": return "N/A"
        return int(v) if float(v).is_integer() else round(v, 4)

    def find_best_ports(app_name, event_name, available_ports):
        if not available_ports: return ["N/A"]
        if len(available_ports) == 1: return [available_ports[0]]
        for p in available_ports:
            if app_name.lower() in p.lower(): return [p]
        for p in available_ports:
            if event_name.lower() in p.lower(): return [p]
        return available_ports 

    # ==========================================
    # MAIN PARSING
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem: continue
            
        mapping_short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', mapping_short_name)
        if not match: continue
            
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        swc_ports = mapping_to_ports.get(mapping_short_name, [])
        
        valid_methods = []
        valuestate_app_name = None
        valuestate_impl_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            impl_ref = dt_map.xpath("*[local-name()='IMPLEMENTATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                impl_path_raw = impl_ref[0].text.split("/")[-1].strip() if impl_ref and impl_ref[0].text else None
                
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    valuestate_impl_name = impl_path_raw
                    continue 
                    
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw, impl_path_raw))
        
        for clean_method, raw_app_name, raw_impl_name in valid_methods:
            actual_ports = find_best_ports(raw_app_name, raw_event_name, swc_ports)
            
            for actual_port_name in actual_ports:
                
                # --- THE NEW SIGNAL STRING PATH LOGIC ---
                # Strip "Interface" AND any trailing numbers (e.g., "Interface1")
                if actual_port_name != "N/A":
                    clean_port = re.sub(r'Interface\d*$', '', actual_port_name, flags=re.IGNORECASE)
                else:
                    clean_port = someip_event # Fallback if no port is mapped
                
                if raw_app_name in records_dict:
                    for element in records_dict[raw_app_name]:
                        elem_name = element["name"]
                        tref = element["tref"] 
                        
                        c_data = get_compu_data(tref)
                        datatype = app_to_basetype.get(tref, "N/A")
                        if datatype == "N/A":
                            clean_match = re.match(r'^(u?s?int(?:8|16|32|64)|float(?:32|64)|boolean|double)', tref, re.IGNORECASE)
                            datatype = clean_match.group(1).lower() if clean_match else "N/A"
                        
                        states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()]) if c_data["has_enums"] else ("Physical Value" if c_data["min"] is not None else "No Data")
                        
                        # NEW CAPL PATH
                        sig_str = f'"EthernetCluster::sif_{sif}::{clean_port}::{elem_name}"'
                        
                        parsed_data.append({
                            "Cluster": "EthernetCluster", "SIF": sif, "Event": someip_event,
                            "Port": actual_port_name, "Method": elem_name, "DataType": datatype,
                            "Enums": states_str, "Min": format_val(c_data["min"]),
                            "Mid": format_val(c_data["mid"]), "Max": format_val(c_data["max"]),
                            "Factor": format_val(c_data["factor"]), "Offset": format_val(c_data["offset"]),
                            "Unit": c_data["unit"], "Signal_String": sig_str
                        })

                else:
                    c_data = get_compu_data(raw_app_name)
                    datatype = impl_to_basetype.get(raw_impl_name, "N/A") if raw_impl_name else "N/A"
                    
                    states_str = " | ".join([f"{k}: {v}" for k, v in c_data["enums"].items()]) if c_data["has_enums"] else ("Physical Value" if c_data["min"] is not None else "No Data")

                    # NEW CAPL PATH
                    sig_str = f'"EthernetCluster::sif_{sif}::{clean_port}::{clean_method}"'
                    
                    parsed_data.append({
                        "Cluster": "EthernetCluster", "SIF": sif, "Event": someip_event,
                        "Port": actual_port_name, "Method": clean_method, "DataType": datatype,
                        "Enums": states_str, "Min": format_val(c_data["min"]),
                        "Mid": format_val(c_data["mid"]), "Max": format_val(c_data["max"]),
                        "Factor": format_val(c_data["factor"]), "Offset": format_val(c_data["offset"]),
                        "Unit": c_data["unit"], "Signal_String": sig_str
                    })
                
                # VALUE_STATE EXPANSION
                if valuestate_app_name and raw_app_name not in records_dict:
                    vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                    vs_method = f"{clean_method}ValueState"
                    
                    vs_c_data = get_compu_data(valuestate_app_name)
                    vs_datatype = impl_to_basetype.get(valuestate_impl_name, "N/A") if valuestate_impl_name else "N/A"
                    vs_states = " | ".join([f"{k}: {v}" for k, v in vs_c_data["enums"].items()]) if vs_c_data["has_enums"] else "Physical Value"
                    
                    # NEW CAPL PATH
                    vs_sig_str = f'"EthernetCluster::sif_{sif}::{clean_port}::{vs_method}"'
                    
                    parsed_data.append({
                        "Cluster": "EthernetCluster", "SIF": sif, "Event": vs_event,
                        "Port": actual_port_name, "Method": vs_method, "DataType": vs_datatype, 
                        "Enums": vs_states, "Min": format_val(vs_c_data["min"]), 
                        "Mid": format_val(vs_c_data["mid"]), "Max": format_val(vs_c_data["max"]), 
                        "Factor": format_val(vs_c_data["factor"]), "Offset": format_val(vs_c_data["offset"]), 
                        "Unit": vs_c_data["unit"], "Signal_String": vs_sig_str
                    })

    df = pd.DataFrame(parsed_data)
    
    if not df.empty:
        # Drop duplicates that might still arise from identical redundant mappings
        df = df.drop_duplicates(subset=['Signal_String']).sort_values(by=['SIF', 'Port', 'Method']).reset_index(drop=True)
    
    return df

In [ ]:
file_name = "ETH_CAN.arxml" 
df_signals = parse_arxml_complete(file_name)

if not df_signals.empty:
    duplicate_signals = df_signals.duplicated(subset=['Signal_String']).sum()
    print(f"Duplicate Signal Strings: {duplicate_signals} ", end="")
    print("(🚨 WARNING!)" if duplicate_signals > 0 else "(✅ CLEAN!)")
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    display(df_signals[['SIF', 'Port', 'Method', 'Signal_String', 'DataType']].sample(15)) # View 15 random signals

Parsing ETH_CAN.arxml (Final Architecture: Exploding Ports & Cleaning Paths)...
Duplicate Signal Strings: 0 (✅ CLEAN!)


,SIF,Port,Method,Signal_String,DataType
6554,662,SomeIpCabinAirFlapStatusEventInterface,cabinAirFlapFrontLeftMixer_Position,"""EthernetCluster::sif_662::SomeIpCabinAirFlapStatusEvent::cabinAirFlapFrontLeftMixer_Position""",uint16
25281,8275,SomeIpWindowsEventInterface,extendedPosition2,"""EthernetCluster::sif_8275::SomeIpWindowsEvent::extendedPosition2""",uint8
29161,8290,setAlarmActivationDisplayInterface,AccessMedium,"""EthernetCluster::sif_8290::setAlarmActivationDisplay::AccessMedium""",uint8
29102,8290,SomeIpWalDisturberRequestAlertEventInterface,AlarmExitDisplayValueState,"""EthernetCluster::sif_8290::SomeIpWalDisturberRequestAlertEvent::AlarmExitDisplayValueState""",uint8
39247,8373,SomeIpExteriorMirrorUserSettingsStateEventInterface,DrivingSideSelectionRequestT1,"""EthernetCluster::sif_8373::SomeIpExteriorMirrorUserSettingsStateEvent::DrivingSideSelectionRequestT1""",uint8
18454,8253,setAccessSilentModeStateInterface,lateralDoorSound,"""EthernetCluster::sif_8253::setAccessSilentModeState::lateralDoorSound""",uint8
30400,8292,setWelcomeGoodbyeSequenceInterface,usageId14ValueState,"""EthernetCluster::sif_8292::setWelcomeGoodbyeSequence::usageId14ValueState""",uint8
29175,8290,setAlarmActivationDisplayInterface,CidMowerAlertRequest,"""EthernetCluster::sif_8290::setAlarmActivationDisplay::CidMowerAlertRequest""",uint8
8885,8240,SomeIpCvSensorPerformanceEventInterface,sensororigin,"""EthernetCluster::sif_8240::SomeIpCvSensorPerformanceEvent::sensororigin""",float32
11436,8243,SomeIphvChgFlapOpenAlertEventInterface,chargerThermalLossToCoolant,"""EthernetCluster::sif_8243::SomeIphvChgFlapOpenAlertEvent::chargerThermalLossToCoolant""",float32


In [ ]:
import pandas as pd
from IPython.display import display

def validate_requirements_robust(req_csv_path, df_signals):
    print(f"Loading Requirements from: {req_csv_path}...")
    
    try:
        df_req = pd.read_csv(req_csv_path)
    except FileNotFoundError:
        print(f"Error: Could not find '{req_csv_path}'")
        return None, None

    # ==========================================
    # 1. NORMALIZE KEYS FOR MATCHING
    # ==========================================
    # Clean up SIF (handle Excel appending '.0') and lower-case the Method
    df_req['match_sif'] = df_req['Service ID'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    df_req['match_method'] = df_req['Attribute value'].astype(str).str.strip().str.lower()
    
    df_signals['match_sif'] = df_signals['SIF'].astype(str).str.strip()
    df_signals['match_method'] = df_signals['Method'].astype(str).str.strip().str.lower()

    # Since we exploded ports, a single SIF+Method might exist across multiple ports.
    # We drop duplicates on SIF+Method just for the merge so we don't accidentally multiply CSV rows.
    df_signals_unique = df_signals.drop_duplicates(subset=['match_sif', 'match_method'])

    # ==========================================
    # 2. PERFORM THE RELAXED MATCH (SIF + Method)
    # ==========================================
    merged_df = pd.merge(
        df_req, 
        df_signals_unique[[
            'match_sif', 'match_method', 'Cluster', 'Event', 'Port', 
            'Signal_String', 'DataType', 'Enums', 
            'Min', 'Mid', 'Max', 'Factor', 'Offset', 'Unit'
        ]], 
        on=['match_sif', 'match_method'], 
        how='left',
        suffixes=('_CSV', '_ARXML')
    )

    merged_df['Is_Found_In_DB'] = merged_df['Signal_String'].notna()

    # ==========================================
    # 3. STATISTICS
    # ==========================================
    total_reqs = len(merged_df)
    matched_reqs = merged_df['Is_Found_In_DB'].sum()
    missing_reqs = total_reqs - matched_reqs
    match_percentage = (matched_reqs / total_reqs) * 100 if total_reqs > 0 else 0

    print("=== FINAL VALIDATION STATISTICS ===")
    print(f"Total Requirements   : {total_reqs}")
    print(f"✅ Matched (SIF+Method): {matched_reqs} ({match_percentage:.2f}%)")
    print(f"❌ Still Missing       : {missing_reqs}\n")

    df_matched = merged_df[merged_df['Is_Found_In_DB'] == True].copy()
    df_missing = merged_df[merged_df['Is_Found_In_DB'] == False].copy()

    # Cleanup temp columns
    df_matched = df_matched.drop(columns=['match_sif', 'match_method', 'Is_Found_In_DB'])
    df_missing = df_missing.drop(columns=['match_sif', 'match_method', 'Is_Found_In_DB'])

    return df_matched, df_missing

# ==========================================
# EXECUTION
# ==========================================
req_file_path = "test_requirements_ E2E_ETH.csv"

df_matched, df_missing = validate_requirements_robust(req_file_path, df_signals)

if df_matched is not None and not df_matched.empty:
    print("--- 🟢 Top 10 SUCCESSFULLY MATCHED Requirements ---")
    
    # Force pandas to show all columns and full strings
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    
    # The ultimate comprehensive view
    display_cols_matched = [
        # --- From the Requirements CSV ---
        'E2E REQ (used for implementation)', 
        'Service ID', 
        'Attribute value', 
        'Port_CSV', 
        
        # --- Extracted from the ARXML ---
        'Port_ARXML',
        'Event',
        'Signal_String', 
        'DataType', 
        'Enums', 
        'Min', 
        'Mid', 
        'Max', 
        'Factor', 
        'Offset', 
        'Unit'
    ]
    
    # Safely display only the columns that exist to prevent KeyErrors
    display(df_matched[[c for c in display_cols_matched if c in df_matched.columns]].head(10))

if df_missing is not None and not df_missing.empty:
    print("\n--- 🔴 Top 10 STILL MISSING Requirements ---")
    display_cols_missing = ['E2E REQ (used for implementation)', 'Service ID', 'Attribute value', 'Port']
    
    missing_cols_to_show = [c for c in display_cols_missing if c in df_missing.columns]
    if 'Port_CSV' in df_missing.columns and 'Port' not in df_missing.columns:
        missing_cols_to_show.append('Port_CSV')
        
    display(df_missing[missing_cols_to_show].head(10))

Loading Requirements from: test_requirements_ E2E_ETH.csv...
=== FINAL VALIDATION STATISTICS ===
Total Requirements   : 2305
✅ Matched (SIF+Method): 2303 (99.91%)
❌ Still Missing       : 2

--- 🟢 Top 10 SUCCESSFULLY MATCHED Requirements ---


,E2E REQ (used for implementation),Service ID,Attribute value,Port_CSV,Port_ARXML,Event,Signal_String,DataType,Available_States,Min,Mid,Max,Factor,Offset,Unit
0,REQ_ETH_E2E_20_1_1,591,absInRegulation,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationStates,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsInRegulation""",uint8,0: ABSINREGULATION_T_NO_ABS_REGULATION | 1: ABSINREGULATION_T_ABS_IN_REGULATION,0,0,1,N/A,N/A,N/A
1,REQ_ETH_E2E_20_2_1,591,absInRegulation,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationStates,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsInRegulation""",uint8,0: ABSINREGULATION_T_NO_ABS_REGULATION | 1: ABSINREGULATION_T_ABS_IN_REGULATION,0,0,1,N/A,N/A,N/A
2,REQ_ETH_E2E_20_3_1,591,absInRegulation,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationStates,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsInRegulation""",uint8,0: ABSINREGULATION_T_NO_ABS_REGULATION | 1: ABSINREGULATION_T_ABS_IN_REGULATION,0,0,1,N/A,N/A,N/A
3,REQ_ETH_E2E_20_1_0,591,absInRegulationValueState,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationState,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsInRegulationValueState""",uint8,0: VALUE_STATE_UNAVAILABLE | 2: VALUE_STATE_INVALID | 1: VALUE_STATE_VALID,0,1,2,N/A,N/A,N/A
4,REQ_ETH_E2E_20_2_0,591,absInRegulationValueState,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationState,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsInRegulationValueState""",uint8,0: VALUE_STATE_UNAVAILABLE | 2: VALUE_STATE_INVALID | 1: VALUE_STATE_VALID,0,1,2,N/A,N/A,N/A
5,REQ_ETH_E2E_20_3_0,591,absInRegulationValueState,ChassisBrakingRegulationStates_ChassisRegulationActivationState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationState,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsInRegulationValueState""",uint8,0: VALUE_STATE_UNAVAILABLE | 2: VALUE_STATE_INVALID | 1: VALUE_STATE_VALID,0,1,2,N/A,N/A,N/A
6,REQ_ETH_E2E_21_1_3,591,absMalfunction,ChassisBrakingRegulationStates_ChassisRegulationMalfunctionState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationStates,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsMalfunction""",uint8,0: ABSMALFUNCTION_T_NO_ABS_MALFUNCTION | 1: ABSMALFUNCTION_T_ABS_MALFUNCTION,0,0,1,N/A,N/A,N/A
7,REQ_ETH_E2E_21_1_2,591,absMalfunctionValueState,ChassisBrakingRegulationStates_ChassisRegulationMalfunctionState,SomeIpChassisRegulationActivationStateInterface,SomeIpChassisBrakingRegulationState,"""EthernetCluster::sif_591::SomeIpChassisRegulationActivationState::AbsMalfunctionValueState""",uint8,0: VALUE_STATE_UNAVAILABLE | 2: VALUE_STATE_INVALID | 1: VALUE_STATE_VALID,0,1,2,N/A,N/A,N/A
8,REQ_ETH_E2E_24_0_1,645,absStateDisplayRequest,ChassisAlerts_BrakeAndSteeringAlerts,SomeIpBrakeAndSteeringAlertsInterface,SomeIpChassisAlerts,"""EthernetCluster::sif_645::SomeIpBrakeAndSteeringAlerts::AbsStateDisplayRequest""",uint8,0: ABSSTATEDISPLAYREQUEST_T_NO_FAILURE | 1: ABSSTATEDISPLAYREQUEST_T_FAILURE | 2: ABSSTATEDISPLAYREQUEST_T_IN_EXT_DIAG_SESSION | 3: ABSSTATEDISPLAYREQUEST_T_TO_BE_CALIBRATED,0,1,3,N/A,N/A,N/A
9,REQ_ETH_E2E_24_0_2,645,absStateDisplayRequestOccurence,ChassisAlerts_BrakeAndSteeringAlerts,SomeIpBrakeAndSteeringAlertsInterface,SomeIpChassisAlerts,"""EthernetCluster::sif_645::SomeIpBrakeAndSteeringAlerts::absStateDisplayRequestOccurence""",uint8,Physical Value,0,127,255,1,0,N/A



--- 🔴 Top 10 STILL MISSING Requirements ---


,E2E REQ (used for implementation),Service ID,Attribute value,Port_CSV
1033,REQ_ETH_E2E_325_1_1,16438,isVehicleUtcRequested,RequestVehicleUtcTime_requestVehicleUtcTime
1405,REQ_ETH_E2E_325_1_0,16438,requestVehicleUtcValueState,RequestVehicleUtcTime_requestVehicleUtcTime
